# DFN Drive Cycle Simulation with Degradation

This notebook demonstrates the `run_drive_cycle_with_degradation` function which combines:
- Drive cycle simulation with the Doyle-Fuller-Newman (DFN) model
- Coupled degradation mechanisms:
  - SEI growth (solvent-diffusion limited)
  - Lithium plating (partially reversible)
  - Particle cracking and swelling
  - Stress-driven loss of active material (LAM)
  - Porosity changes from side reactions

## Features
- All standard drive cycle analysis (energy, range, temperature)
- Real-time degradation tracking
- Multiple degradation modes (LLI, LAM)
- Porosity evolution monitoring

In [ ]:
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from model_library import run_drive_cycle_with_degradation

print("Imports successful!")
print("✓ Unified function: run_drive_cycle_with_degradation")
print("  - Single-cycle: max_cycles=1 or not specified")
print("  - Multi-cycle:  max_cycles>1 (flat simulation with repeated drive cycles)")
print("\n✓ Function consolidation complete:")
print("  - All logic inline in run_drive_cycle_with_degradation()")
print("  - max_cycles=1 is special case of multi-cycle framework")
print("  - Single entry point for all simulation modes")


ImportError: cannot import name 'print_drive_cycle_degradation_report' from 'model_library.dfn_drive_degradation' (/Users/manik/Github/model_library/src/model_library/dfn_drive_degradation.py)

## 1. Load Drive Cycle

We'll use an automotive drive cycle to demonstrate degradation over time.

In [ ]:
# Available drive cycles:
# - Auto WLTP, Auto US06, Track Nurburgring
# - Aero Quad Drone, Aero UAV, Aero eVTOL

drive_cycle_name = "Auto WLTP"
drive_cycle_path = Path(f"../drive_cycles/{drive_cycle_name} Power.csv")
drive_cycle_df = pd.read_csv(drive_cycle_path)

print(f"Drive cycle: {drive_cycle_name}")
print(
    f"Duration: {drive_cycle_df['time (s)'].max():.0f} s ({drive_cycle_df['time (s)'].max()/60:.1f} min)"
)
print(
    f"Power range: {drive_cycle_df['power (W)'].min():.3f} to {drive_cycle_df['power (W)'].max():.3f} W"
)

# Plot
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(
    drive_cycle_df["time (s)"] / 60,
    drive_cycle_df["power (W)"],
    "b-",
    linewidth=0.5,
)
ax.fill_between(
    drive_cycle_df["time (s)"] / 60, drive_cycle_df["power (W)"], 0, alpha=0.3
)
ax.set_xlabel("Time [min]")
ax.set_ylabel("Power [W]")
ax.set_title(f"{drive_cycle_name} Drive Cycle")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Load Cell Design

In [ ]:
manifest_path = Path("../cells/Tesla_Model3_Prismatic_160Ah_manifest.json")
material_path = Path("../materials")

with open(manifest_path, "r") as f:
    cell_design_manifest = json.load(f)

    for component in ["separator", "electrolyte"]:
        material_name = cell_design_manifest["cell_design"][component]["material"][
            "type"
        ]
        with open(material_path / f"{material_name}.json", "r") as mf:
            cell_design_manifest["cell_design"][component]["material"] = json.load(mf)

    for component in ["negative_electrode", "positive_electrode"]:
        material_name = cell_design_manifest["cell_design"][component]["coating"][
            "formulation"
        ]["primary_active_material"]["name"]
        with open(material_path / f"{material_name}.json", "r") as mf:
            cell_design_manifest["cell_design"][component]["material"] = json.load(mf)

cell_design = cell_design_manifest["cell_design"]
cell_design["nominal_capacity"] = {
    "value": cell_design_manifest["kpis"]["nominal_capacity"]["value"],
    "unit": "Ah",
}
cell_design["nominal_energy"] = {
    "value": cell_design_manifest["kpis"]["nominal_energy"]["value"],
    "unit": "Wh",
}
cell_design["cell_volume"] = {
    "value": cell_design_manifest["kpis"]["cell_volume"]["value"],
    "unit": "cm3",
}

print(f"Cell: {manifest_path.stem}")
print(f"Capacity: {cell_design['nominal_capacity']['value']:.1f} Ah")
print(f"Energy: {cell_design['nominal_energy']['value']:.1f} Wh")

## 3. Configure Simulation with Degradation Options

The DFN model with degradation requires:
- Higher mesh resolution (especially in particles)
- Degradation model selection
- Optional: O'Kane2022 parameter set for validated degradation parameters

In [ ]:
# Prepare config with degradation options
config = {
    # Standard thermal/electrical parameters
    "ambient_temperature_K": 298.15,
    "initial_temperature_K": 298.15,
    "initial_soc": 0.80,
    "upper_voltage_cutoff_V": cell_design["upper_voltage_cutoff"]["value"],
    "lower_voltage_cutoff_V": cell_design["lower_voltage_cutoff"]["value"],
    "contact_resistance_Ohm": 0.0001,
    "total_heat_transfer_coefficient_W_m2K": 10,
    "cooling_surface_area_m2": 0.01,
    "period": "1 second",
    "min_soc": 0.10,
    "max_soc": 0.90,
    # Drive cycle data
    "drive_cycle": {
        "time_s": drive_cycle_df["time (s)"].values,
        "power_W": -drive_cycle_df["power (W)"].values,  # Negative = discharge
        "label": drive_cycle_name,
    },
    # Degradation options
    "use_okane2022_params": False,  # Set True to use validated degradation params
    "skip_capacity_calibration": False,  # Set True for faster testing
    "sei_model": "solvent-diffusion limited",
    # "sei_model": "reaction limited",
    "sei_porosity_change": "true",
    "lithium_plating": "partially reversible",
    "lithium_plating_porosity_change": "true",
    "particle_mechanics": (
        "swelling and cracking",
        "swelling only",
    ),  # (negative, positive)
    "sei_on_cracks": "true",
    "loss_of_active_material": "stress-driven",
    # SEI parameters (required for degradation models)
    "initial_sei_thickness_m": 5e-9,  # 5 nm initial SEI thickness
    "sei_reaction_exchange_current_density_A_m2": 1.5e-7,  # SEI reaction exchange current density [A.m-2]
    "sei_partial_molar_volume_m3_mol": 9.585e-5,  # SEI partial molar volume [m3.mol-1]
    "sei_resistivity_Ohm_m": 2.5e5,  # SEI resistivity [Ohm.m]
    "sei_growth_activation_energy_J_mol": 5e4,  # SEI growth activation energy [J.mol-1]
    "sei_solvent_diffusivity_m2_s": 2.5e-22,  # SEI solvent diffusivity [m2.s-1]
    "bulk_solvent_concentration_mol_m3": 2000.0,  # Bulk solvent concentration [mol.m-3]
    # Lithium plating parameters
    "typical_plated_lithium_concentration_mol_m3": 1000.0,  # Typical plated Li concentration [mol.m-3]
    "lithium_metal_partial_molar_volume_m3_mol": 1.3e-5,  # Li metal partial molar volume [m3.mol-1]
    "exchange_current_density_for_stripping_A_m2": 0.001,  # Exchange-current density for stripping [A.m-2]
    "exchange_current_density_for_plating_A_m2": 0.001,  # Exchange-current density for plating [A.m-2]
    "lithium_plating_transfer_coefficient": 0.5,  # Lithium plating transfer coefficient
    # Mesh points (higher resolution for DFN with degradation)
    "var_pts": {
        "x_n": 10,  # negative electrode spatial
        "x_s": 10,  # separator spatial
        "x_p": 10,  # positive electrode spatial
        "r_n": 30,  # negative particle radial (high for cracking)
        "r_p": 30,  # positive particle radial
    },
}

print("Configuration ready!")
print(f"\nDegradation mechanisms enabled:")
print(f"  - SEI: {config['sei_model']}")
print(f"  - Lithium plating: {config['lithium_plating']}")
print(f"  - Particle mechanics: {config['particle_mechanics']}")
print(f"  - SEI on cracks: {config['sei_on_cracks']}")
print(f"  - Loss of active material: {config['loss_of_active_material']}")
print(f"\nDegradation parameters:")
print(f"  - Initial SEI thickness: {config['initial_sei_thickness_m']*1e9:.1f} nm")
print(f"  - SEI partial molar volume: {config['sei_partial_molar_volume_m3_mol']:.6e} m³/mol")
print(f"  - SEI resistivity: {config['sei_resistivity_Ohm_m']:.2e} Ohm·m")
print(f"  - SEI growth activation energy: {config['sei_growth_activation_energy_J_mol']:.2e} J/mol")
print(f"  - SEI solvent diffusivity: {config['sei_solvent_diffusivity_m2_s']:.2e} m²/s")
print(f"  - Bulk solvent concentration: {config['bulk_solvent_concentration_mol_m3']:.1f} mol/m³")
print(f"  - Plated Li concentration: {config['typical_plated_lithium_concentration_mol_m3']:.1f} mol/m³")
print(f"  - Exchange-current density (plating): {config['exchange_current_density_for_plating_A_m2']:.6f} A/m²")
print(f"  - Exchange-current density (stripping): {config['exchange_current_density_for_stripping_A_m2']:.6f} A/m²")

print(f"  - Li plating transfer coefficient: {config['lithium_plating_transfer_coefficient']:.2f}")
# print(f"  - Li stripping transfer coefficient: {config['lithium_stripping_transfer_coefficient']:.2f}")

## 4. Run Simulation

**Note**: This will take longer than SPMe simulations due to:
- DFN model complexity (solves in both electrode and electrolyte)
- Degradation equations
- Higher mesh resolution
- Capacity calibration (if enabled)

### Single vs Multi-Cycle Time Control

**Single Cycle**: Time is controlled by drive cycle duration (e.g., 60 min for WLTP)
**Multi-Cycle**: Use `max_simulation_time_s` parameter to limit total simulation wall-clock time

Example configurations:
- **Quick test**: `max_cycles=10, max_simulation_time_s=600` (10 cycles or 10 minutes, whichever comes first)
- **1-hour run**: `max_cycles=100, max_simulation_time_s=3600` (stop after 1 hour)
- **Unlimited**: `max_cycles=1000, max_simulation_time_s=None` (run until SoH threshold or max cycles)


In [ ]:
print("Starting DFN simulation with degradation...")
print("This may take several minutes...\n")

result = run_drive_cycle_with_degradation(
    cell_design=cell_design, simulation_config=config
)

if result["success"]:
    print("\n✓ Simulation completed successfully!")
else:
    print(f"\n✗ Simulation failed: {result.get('error')}")

## 5. Results Report

The report includes standard drive cycle metrics plus degradation analysis.

In [ ]:
if result["success"]:
    print("=" * 80)
    print("SIMULATION RESULTS")
    print("=" * 80)
    
    summary = result.get("summary", {})
    config = result.get("config", {})
    
    print("\n📋 Simulation Configuration:")
    print(f"  Drive cycle: {config.get('drive_cycle', {}).get('label', 'unknown')}")
    print(f"  Max cycles: {config.get('max_cycles', 1)}")
    print(f"  SoH threshold: {config.get('soh_threshold', 80.0)}%")
    
    print("\n📊 Results Summary:")
    print(f"  Total cycles completed: {summary.get('total_cycles', 0)}")
    print(f"  Final SoH: {summary.get('final_soh_pct', 0):.2f}%")
    print(f"  Final capacity: {summary.get('final_capacity_Ah', 0):.4f} Ah")
    print(f"  Nominal capacity: {summary.get('nominal_capacity_Ah', 0):.2f} Ah")
    print(f"  Capacity fade: {summary.get('capacity_fade_Ah', 0):.6f} Ah ({summary.get('capacity_fade_pct', 0):.2f}%)")
    print(f"  Total FCE: {summary.get('total_fce', 0):.2f}")
    print(f"  Total throughput: {summary.get('total_throughput_Ah', 0):.2f} Ah")
    print(f"  Total energy throughput: {summary.get('total_energy_throughput_kWh', 0):.2f} kWh")
    print(f"  Total simulation time: {summary.get('total_simulation_time_s', 0):.1f} s ({summary.get('total_simulation_time_hr', 0):.2f} hr)")
    print(f"  Wall-clock time: {summary.get('wall_clock_time_s', 0):.1f} s")
    print(f"  Stop reason: {summary.get('stop_reason', 'unknown')}")
    
    print("\n🔬 Degradation Summary:")
    print(f"  Final LLI: {summary.get('final_LLI_pct', 0):.6f}%")
    print(f"  Final LAM (negative): {summary.get('final_LAM_neg_pct', 0):.6f}%")
    print(f"  Final LAM (positive): {summary.get('final_LAM_pos_pct', 0):.6f}%")
    
    print("\n" + "=" * 80)
else:
    print(f"❌ Simulation failed: {result.get('error')}")


## 6. Degradation Data Access

In [ ]:
if result["success"]:
    print("Result keys:", list(result.keys()))
    print("\nTimeseries arrays:", list(result["timeseries"].keys()))

    if "degradation" in result["timeseries"]:
        print("\nDegradation timeseries:", list(result["timeseries"]["degradation"].keys()))

    if "degradation_summary" in result:
        print("\nDegradation summary:")
        for key, value in result["degradation_summary"].items():
            print(f"  {key}: {value}")

## 7. Visualizations

### Standard Drive Cycle Plots

In [ ]:
if result["success"]:
    ts = result["timeseries"]

    fig, axes = plt.subplots(5, 1, figsize=(14, 10), sharex=True)

    axes[0].plot(ts["time_s"] / 60, ts["current_A"], "b-", lw=0.5)
    axes[0].fill_between(ts["time_s"] / 60, ts["current_A"], 0, alpha=0.3)
    axes[0].set_ylabel("Current [A]")
    axes[0].set_title(f"{drive_cycle_name} Simulation with Degradation (DFN Model)")
    axes[0].grid(True, alpha=0.3)

    axes[1].plot(ts["time_s"] / 60, ts["voltage_V"], "g-", lw=1)
    axes[1].set_ylabel("Voltage [V]")
    axes[1].grid(True, alpha=0.3)

    axes[2].plot(ts["time_s"] / 60, ts["power_W"] / 1000, "orange", lw=0.5)
    axes[2].fill_between(ts["time_s"] / 60, ts["power_W"] / 1000, 0, alpha=0.3, color="orange")
    axes[2].set_ylabel("Power [kW]")
    axes[2].grid(True, alpha=0.3)

    axes[3].plot(ts["time_s"] / 60, ts["soc"] * 100, "k-", lw=1)
    axes[3].set_ylabel("SOC [%]")
    axes[3].grid(True, alpha=0.3)

    axes[4].plot(ts["time_s"] / 60, ts["temperature_K"] - 273.15, "r-", lw=1)
    axes[4].set_xlabel("Time [min]")
    axes[4].set_ylabel("Temperature [°C]")
    axes[4].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

### Degradation Mechanism Plots

In [ ]:
if result["success"] and "degradation" in result["timeseries"]:
    deg = result["timeseries"]["degradation"]
    time_min = ts["time_s"] / 60
    throughput = deg["throughput_Ah"]

    fig, axes = plt.subplots(2, 2, figsize=(14, 10))

    # Loss of Lithium Inventory (LLI) Contributors
    axes[0, 0].plot(throughput, deg["Q_SEI_Ah"], label="SEI", linestyle="dashed")
    axes[0, 0].plot(
        throughput, deg["Q_SEI_cracks_Ah"], label="SEI on cracks", linestyle="dashdot"
    )
    axes[0, 0].plot(
        throughput, deg["Q_plating_Ah"], label="Li plating", linestyle="dotted"
    )
    axes[0, 0].plot(
        throughput,
        deg["Q_side_reactions_Ah"],
        label="All side reactions",
        linewidth=2,
    )
    axes[0, 0].set_xlabel("Throughput capacity [A.h]")
    axes[0, 0].set_ylabel("Capacity loss [A.h]")
    axes[0, 0].set_title("Loss of Lithium Inventory Mechanisms")
    axes[0, 0].legend()
    axes[0, 0].grid(True, alpha=0.3)

    # Major Degradation Modes
    axes[0, 1].plot(throughput, deg["LLI_pct"], label="LLI", linewidth=2)
    axes[0, 1].plot(throughput, deg["LAM_neg_pct"], label="LAM (negative)", linewidth=2)
    axes[0, 1].plot(throughput, deg["LAM_pos_pct"], label="LAM (positive)", linewidth=2)
    axes[0, 1].set_xlabel("Throughput capacity [A.h]")
    axes[0, 1].set_ylabel("Degradation [%]")
    axes[0, 1].set_title("Major Degradation Modes")
    axes[0, 1].legend()
    axes[0, 1].grid(True, alpha=0.3)

    # Porosity Evolution
    axes[1, 0].plot(throughput, deg["porosity_neg"], label="Negative electrode", linewidth=2)
    axes[1, 0].plot(throughput, deg["porosity_pos"], label="Positive electrode", linewidth=2)
    axes[1, 0].set_xlabel("Throughput capacity [A.h]")
    axes[1, 0].set_ylabel("Porosity")
    axes[1, 0].set_title("Electrode Porosity Evolution")
    axes[1, 0].legend()
    axes[1, 0].grid(True, alpha=0.3)

    # Lithium Lost
    axes[1, 1].plot(throughput, deg["Li_lost_mol"], "b-", linewidth=2)
    axes[1, 1].set_xlabel("Throughput capacity [A.h]")
    axes[1, 1].set_ylabel("Total lithium lost [mol]")
    axes[1, 1].set_title("Total Lithium Lost")
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle("Degradation Analysis", fontsize=16, fontweight="bold")
    plt.tight_layout()
    plt.show()
else:
    print("No degradation data available")

### Combined Performance and Degradation

In [ ]:
if result["success"] and "degradation" in result["timeseries"]:
    deg = result["timeseries"]["degradation"]
    time_min = ts["time_s"] / 60

    fig, axes = plt.subplots(4, 1, figsize=(14, 10), sharex=True)

    # Voltage
    axes[0].plot(time_min, ts["voltage_V"], "g-", lw=1)
    axes[0].set_ylabel("Voltage [V]")
    axes[0].set_title("Performance + Degradation Over Time")
    axes[0].grid(True, alpha=0.3)

    # SOC
    axes[1].plot(time_min, ts["soc"] * 100, "k-", lw=1)
    axes[1].set_ylabel("SOC [%]")
    axes[1].grid(True, alpha=0.3)

    # LLI and LAM
    axes[2].plot(time_min, deg["LLI_pct"], label="LLI", linewidth=1.5)
    axes[2].plot(time_min, deg["LAM_neg_pct"], label="LAM (neg)", linewidth=1.5)
    axes[2].plot(time_min, deg["LAM_pos_pct"], label="LAM (pos)", linewidth=1.5)
    axes[2].set_ylabel("Degradation [%]")
    axes[2].legend(loc="upper left")
    axes[2].grid(True, alpha=0.3)

    # Porosity
    axes[3].plot(time_min, deg["porosity_neg"], label="Neg electrode", linewidth=1.5)
    axes[3].plot(time_min, deg["porosity_pos"], label="Pos electrode", linewidth=1.5)
    axes[3].set_xlabel("Time [min]")
    axes[3].set_ylabel("Porosity")
    axes[3].legend(loc="upper right")
    axes[3].grid(True, alpha=0.3)

    plt.tight_layout()
    plt.show()

## 8. Comparison: Impact of Single Drive Cycle

This shows the degradation from a single drive cycle. For meaningful degradation, you would typically need to simulate many cycles.

In [ ]:
if result["success"] and "degradation_summary" in result:
    deg_sum = result["degradation_summary"]

    print("="*70)
    print("SINGLE CYCLE DEGRADATION IMPACT")
    print("="*70)
    print(f"\nThroughput: {deg_sum['throughput_final_Ah']:.2f} A.h")
    print(f"\nCapacity fade:")
    print(f"  From SEI:           {deg_sum['Q_SEI_final_Ah']:.6f} A.h")
    print(f"  From SEI on cracks: {deg_sum['Q_SEI_cracks_final_Ah']:.6f} A.h")
    print(f"  From Li plating:    {deg_sum['Q_plating_final_Ah']:.6f} A.h")
    print(f"  Total side rxns:    {deg_sum['Q_side_reactions_final_Ah']:.6f} A.h")

    print(f"\nDegradation modes:")
    print(f"  LLI:         {deg_sum['LLI_final_pct']:.6f}%")
    print(f"  LAM (neg):   {deg_sum['LAM_neg_final_pct']:.6f}%")
    print(f"  LAM (pos):   {deg_sum['LAM_pos_final_pct']:.6f}%")

    print(f"\nPorosity change (negative electrode):")
    print(f"  Initial: {deg_sum['porosity_neg_initial']:.6f}")
    print(f"  Final:   {deg_sum['porosity_neg_final']:.6f}")
    print(f"  Change:  {deg_sum['porosity_neg_change']:.6f}")

    print(f"\nNote: Degradation from a single cycle is typically very small.")
    print(f"      Run multiple cycles or use ageing protocols for significant degradation.")
    print("="*70)

## 9. Optional: Simplified Configuration for Faster Testing

For debugging or quick tests, you can disable certain features:

In [ ]:
# Fast test configuration (uncomment to use)
fast_config = {
    **config,
    "skip_capacity_calibration": False,  # Skip calibration
    "sei_model": "reaction limited",  # Use simpler SEI model"
    "sei_porosity_change": "false",  # Disable SEI porosity change
    "lithium_plating": "none",  # Disable lithium plating
    "particle_mechanics": "none",  # Disable particle cracking
    "sei_on_cracks": "false",  # Disable SEI on cracks
    "loss_of_active_material": "none",  # Disable LAM
    "var_pts": {
        "x_n": 5,
        "x_s": 5,
        "x_p": 5,
        "r_n": 10,  # Lower resolution
        "r_p": 10,
    },
}

In [ ]:
fast_result = run_drive_cycle_with_degradation(
    cell_design=cell_design, simulation_config=fast_config
)

## Summary

This notebook demonstrated:
1. **DFN model setup** with coupled degradation mechanisms
2. **Drive cycle simulation** with real-time degradation tracking
3. **Degradation analysis** including LLI, LAM, and porosity changes
4. **Visualization** of both performance and degradation metrics

### Key Takeaways:
- DFN simulations are more computationally intensive than SPMe
- Single-cycle degradation is typically very small
- Degradation becomes significant over hundreds/thousands of cycles
- Porosity changes can trigger early failure modes
- Multiple degradation mechanisms interact and compound

### Next Steps:
- Simulate multiple drive cycles in sequence
- Compare different degradation parameter sets
- Analyze temperature effects on degradation
- Optimize drive profiles to minimize degradation

## 10. Multi-Cycle Simulation (Flat Implementation)

The unified `run_drive_cycle_with_degradation` function automatically detects multi-cycle mode when `max_cycles > 1`.

### Implementation: ONE Flat Simulation
Instead of running N separate simulations (recursion/loops), this creates **one PyBaMM experiment with N repeated drive cycle steps**, simulated continuously with degradation evolution. Much more efficient!

### Key Configuration Parameters:
All control parameters go in the `simulation_config`:
- **`max_cycles`**: Number of drive cycle repetitions (triggers multi-cycle mode if > 1)
- **`soh_threshold`**: Stop when SoH drops below this (%, default: 80.0)
- **`max_simulation_time_s`**: Total sim time limit in seconds (default: None = unlimited)
- **`save_interval`**: Sampling interval for cycle history (default: 10)
- **Degradation options**: `sei_model`, `lithium_plating`, etc. (defaults to SEI-only)

### Usage Example:
```python
config = {
    # Drive cycle and cell parameters...
    # Degradation options
    "sei_model": "solvent-diffusion limited",
    "lithium_plating": "partially reversible",
    # Multi-cycle control (triggers flat multi-cycle if max_cycles > 1)
    "max_cycles": 100,
    "soh_threshold": 80.0,
    "max_simulation_time_s": 3600,  # 1 hour limit
    "save_interval": 5,
}

# Same function for single or multi-cycle!
result = run_drive_cycle_with_degradation(
    cell_design=cell_design,
    simulation_config=config,
)
```

### Outputs:
- **cycle_history**: Sampled metrics at save_interval
- **summary**: Final statistics with stop reason
- **timeseries**: Full continuous degradation data
- **stop_reason**: 'soh_threshold', 'max_cycles', 'time_limit', or 'completed'


### Run Multi-Cycle Simulation

The unified `run_drive_cycle_with_degradation` function uses **one flat simulation** with repeated drive cycles:

**All parameters go in `simulation_config`**:
- `max_cycles`: Number of drive cycle repetitions (> 1 triggers multi-cycle mode)
- `soh_threshold`: Stop when capacity drops below target (default: 80%)
- `max_simulation_time_s`: Total wall-clock time limit (default: None = no limit)
- `save_interval`: Sampling interval for history (default: 10)
- Degradation options: `sei_model`, `lithium_plating`, etc.

**Simple Usage** (same function for single or multi-cycle!):
```python
config = {
    # ... drive cycle and cell parameters ...
    # Degradation
    "sei_model": "solvent-diffusion limited",
    "lithium_plating": "partially reversible",
    # Multi-cycle control
    "max_cycles": 100,  # Triggers flat multi-cycle simulation
    "soh_threshold": 80.0,
    "max_simulation_time_s": 3600,  # 1 hour
    "save_interval": 5,
}

result = run_drive_cycle_with_degradation(cell_design, config)
```

**Performance**: ONE continuous PyBaMM simulation with degradation evolution - no recursion or loops!


### Example Degradation Configurations

The function defaults to **SEI-only** degradation. You can enable additional mechanisms by specifying them in your config:

In [ ]:
# Example 1: SEI-only (DEFAULT - no need to specify)
sei_only_config = {
    **fast_config,
    # Multi-cycle control parameters
    "soh_threshold": 80.0,
    "max_cycles": 100,
    "max_simulation_time_s": None,  # No time limit
    "save_interval": 5,
}

# Example 2: SEI + Lithium Plating
sei_plating_config = {
    **fast_config,
    "lithium_plating": "partially reversible",
    "lithium_plating_porosity_change": "true",
    # Multi-cycle control
    "soh_threshold": 80.0,
    "max_cycles": 100,
    "max_simulation_time_s": 3600,  # 1 hour
    "save_interval": 5,
}

# Example 3: SEI + Particle Mechanics (no LAM)
sei_mechanics_config = {
    **fast_config,
    "particle_mechanics": ("swelling and cracking", "swelling only"),
    "sei_on_cracks": "true",
    # Multi-cycle control
    "soh_threshold": 80.0,
    "max_cycles": 50,
    "max_simulation_time_s": 7200,  # 2 hours
    "save_interval": 10,
}

# Example 4: Full Degradation (ALL mechanisms)
full_degradation_config = {
    **fast_config,
    "sei_model": "solvent-diffusion limited",
    "sei_porosity_change": "true",
    "lithium_plating": "partially reversible",
    "lithium_plating_porosity_change": "true",
    "particle_mechanics": ("swelling and cracking", "swelling only"),
    "sei_on_cracks": "true",
    "loss_of_active_material": "stress-driven",
    # Multi-cycle control
    "soh_threshold": 80.0,
    "max_cycles": 200,
    "max_simulation_time_s": None,  # No time limit
    "save_interval": 20,
}

print("Degradation configuration examples created!")
print("\n📋 Available configs:")
print("  • sei_only_config         - SEI growth only (fastest)")
print("  • sei_plating_config      - SEI + Li plating (1 hour limit)")
print("  • sei_mechanics_config    - SEI + particle cracking (2 hour limit)")
print("  • full_degradation_config - All mechanisms (no time limit)")


### Time-Controlled Simulation Examples

Different strategies for controlling simulation duration:


In [ ]:
# Time-controlled simulation configs
# All parameters go directly in the simulation_config

# Example A: Quick 10-minute test
quick_test_config = {
    **fast_config,
    "soh_threshold": 80.0,
    "max_cycles": 2,
    "max_simulation_time_s": 600,  # 10 minutes
    "save_interval": 2,
}

# Example B: 1-hour simulation
one_hour_config = {
    **fast_config,
    "lithium_plating": "partially reversible",
    "lithium_plating_porosity_change": "true",
    "soh_threshold": 80.0,
    "max_cycles": 100,
    "max_simulation_time_s": 3600,  # 1 hour
    "save_interval": 5,
}

# Example C: 2-hour simulation
two_hour_config = {
    **fast_config,
    "lithium_plating": "partially reversible",
    "lithium_plating_porosity_change": "true",
    "soh_threshold": 80.0,
    "max_cycles": 200,
    "max_simulation_time_s": 7200,  # 2 hours
    "save_interval": 10,
}

# Example D: Unlimited time (run to completion)
unlimited_config = {
    **fast_config,
    "soh_threshold": 80.0,
    "max_cycles": 1000,
    "max_simulation_time_s": None,  # No time limit
    "save_interval": 20,
}

print("Time-controlled simulation configs created!")
print("\n⏱️ Available configurations:")
print("  • quick_test_config  - 10 cycles or 10 min (fastest)")
print("  • one_hour_config    - Up to 100 cycles in 1 hour")
print("  • two_hour_config    - Up to 200 cycles in 2 hours")
print("  • unlimited_config   - Run until SoH < 80% or 1000 cycles")
print("\n✓ All control parameters are in the config dict")
print("✓ Use: run_multi_cycle_degradation(cell_design, config)")


In [ ]:
# Run multi-cycle simulation using the UNIFIED function
# All parameters (max_cycles, max_simulation_time_s, soh_threshold, save_interval) 
# are included in the simulation_config
# 
# Implementation: ONE flat PyBaMM simulation with repeated drive cycles
# (no recursion, no loops, just periodic drive cycle cutoff by total time or FCE)

# Use the unified function (auto-detects multi-cycle from max_cycles > 1)
multi_cycle_results = run_drive_cycle_with_degradation(
    cell_design=cell_design,
    simulation_config=quick_test_config,  # Contains max_cycles > 1
)

# Print summary
print_multi_cycle_summary(multi_cycle_results)











































































#################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################################### Summary Report: FCE, Energy Throughput, and SoH

In [ ]:
# Print comprehensive summary
if multi_cycle_results.get("success"):
    summary = multi_cycle_results["summary"]
    
    print("=" * 80)
    print("MULTI-CYCLE DEGRADATION SUMMARY")
    print("=" * 80)
    print(f"\n📊 Performance Metrics:")
    print(f"  Total cycles: {summary['total_cycles']}")
    print(f"  Final SoH: {summary['final_soh_pct']:.2f}%")
    print(f"  Final capacity: {summary['final_capacity_Ah']:.4f} Ah")
    print(f"  Capacity fade: {summary['capacity_fade_Ah']:.6f} Ah ({summary['capacity_fade_pct']:.2f}%)")
    print(f"  Total FCE: {summary['total_fce']:.2f}")
    print(f"  Total energy throughput: {summary['total_energy_throughput_kWh']:.2f} kWh")
    print(f"  Total throughput Ah: {summary['total_throughput_Ah']:.2f} Ah")
    print(f"  Total simulation time: {summary['total_simulation_time_hr']:.2f} hours")
    print(f"  Wall-clock time: {summary['wall_clock_time_s']:.1f} s")
    print(f"  Stop reason: {summary['stop_reason']}")
    
    print(f"\n🔬 Degradation Mechanisms:")
    print(f"  LLI (Loss of Lithium Inventory): {summary['final_LLI_pct']:.6f}%")
    print(f"  LAM negative electrode: {summary['final_LAM_neg_pct']:.6f}%")
    print(f"  LAM positive electrode: {summary['final_LAM_pos_pct']:.6f}%")
    
    print(f"\n? Additional Details:")
    print(f"  Average time per cycle: {summary['total_simulation_time_s'] / summary['total_cycles']:.1f} s")
    print(f"  Threshold reached: {summary['threshold_reached']}")
    
    print("=" * 80)
else:
    print(f"❌ Simulation failed: {multi_cycle_results.get('error')}")


### Degradation Evolution Plots

In [ ]:
history = multi_cycle_results["cycle_history"]

fig, axes = plt.subplots(3, 2, figsize=(16, 12))

# 1. SoH vs Cycles
axes[0, 0].plot(history["cycle_number"], history["soh_pct"], 'b-', linewidth=2)
axes[0, 0].axhline(y=80, color='r', linestyle='--', label='80% threshold')
axes[0, 0].set_xlabel("Cycle Number")
axes[0, 0].set_ylabel("State of Health [%]")
axes[0, 0].set_title("SoH Degradation Over Cycles")
axes[0, 0].grid(True, alpha=0.3)
axes[0, 0].legend()

# 2. SoH vs FCE
axes[0, 1].plot(history["fce"], history["soh_pct"], 'g-', linewidth=2)
axes[0, 1].axhline(y=80, color='r', linestyle='--', label='80% threshold')
axes[0, 1].set_xlabel("Full Cycle Equivalent (FCE)")
axes[0, 1].set_ylabel("State of Health [%]")
axes[0, 1].set_title("SoH vs FCE")
axes[0, 1].grid(True, alpha=0.3)
axes[0, 1].legend()

# 3. SoH vs Energy Throughput
axes[1, 0].plot(np.array(history["energy_throughput_Wh"]) / 1000, 
                history["soh_pct"], 'orange', linewidth=2)
axes[1, 0].axhline(y=80, color='r', linestyle='--', label='80% threshold')
axes[1, 0].set_xlabel("Energy Throughput [kWh]")
axes[1, 0].set_ylabel("State of Health [%]")
axes[1, 0].set_title("SoH vs Energy Throughput")
axes[1, 0].grid(True, alpha=0.3)
axes[1, 0].legend()

# 4. Degradation Modes vs Cycles
axes[1, 1].plot(history["cycle_number"], history["LLI_pct"], 
                label='LLI', linewidth=2)
axes[1, 1].plot(history["cycle_number"], history["LAM_neg_pct"], 
                label='LAM (neg)', linewidth=2)
axes[1, 1].plot(history["cycle_number"], history["LAM_pos_pct"], 
                label='LAM (pos)', linewidth=2)
axes[1, 1].set_xlabel("Cycle Number")
axes[1, 1].set_ylabel("Degradation [%]")
axes[1, 1].set_title("Degradation Modes Evolution")
axes[1, 1].legend()
axes[1, 1].grid(True, alpha=0.3)

# 5. Capacity Loss Mechanisms vs FCE
axes[2, 0].plot(history["fce"], history["Q_SEI_Ah"], 
                label='SEI', linewidth=2, linestyle='--')
axes[2, 0].plot(history["fce"], history["Q_SEI_cracks_Ah"], 
                label='SEI on cracks', linewidth=2, linestyle='-.')
axes[2, 0].plot(history["fce"], history["Q_plating_Ah"], 
                label='Li plating', linewidth=2, linestyle=':')
axes[2, 0].set_xlabel("Full Cycle Equivalent (FCE)")
axes[2, 0].set_ylabel("Capacity Loss [Ah]")
axes[2, 0].set_title("Capacity Loss Mechanisms")
axes[2, 0].legend()
axes[2, 0].grid(True, alpha=0.3)

# 6. Porosity Evolution vs FCE
axes[2, 1].plot(history["fce"], history["porosity_neg"], 
                label='Negative electrode', linewidth=2)
axes[2, 1].plot(history["fce"], history["porosity_pos"], 
                label='Positive electrode', linewidth=2)
axes[2, 1].set_xlabel("Full Cycle Equivalent (FCE)")
axes[2, 1].set_ylabel("Porosity")
axes[2, 1].set_title("Electrode Porosity Evolution")
axes[2, 1].legend()
axes[2, 1].grid(True, alpha=0.3)

plt.suptitle("Multi-Cycle Degradation Analysis", fontsize=16, fontweight='bold')
plt.tight_layout()
plt.show()

### Export Results to DataFrame

In [ ]:
# Create DataFrame from cycle history
df_cycles = pd.DataFrame(multi_cycle_results["cycle_history"])

# Display first and last few cycles
print("\n📊 First 5 cycles:")
print(df_cycles.head().to_string(index=False))

print("\n📊 Last 5 cycles:")
print(df_cycles.tail().to_string(index=False))

# Save to CSV (optional)
# df_cycles.to_csv("multi_cycle_degradation_results.csv", index=False)
# print("\n✓ Results saved to multi_cycle_degradation_results.csv")